# Dhaka Flex Plus — Build a test APK without GitHub Actions

This notebook runs entirely in a temporary **Google Colab CPU runtime**. It checks out the current `main` branch of [`fantominsider/dhaka-flex-plus`](https://github.com/fantominsider/dhaka-flex-plus), installs Java 17, official Android API 36 SDK command-line tools, and Gradle 8.11.1, then attempts to build `:app:assembleDebug`.

**Instructions:** Sign into a Google account, choose **Runtime → Run all**, approve notebook execution, and download the APK if the build passes. **No payment card, GitHub Actions, or Google Drive mounting is needed.**

**Warning:** The application is an unverified open-source development build, not a certified APK. The first attempt may reveal Java, LibVLC, or Android SDK compilation issues. On failure the notebook downloads `DhakaFlexPlus-Colab-Build.log`—share that log for debugging. Google Colab free capacity and session access are subject to availability. Only install APKs you trust, and test streaming against servers you are authorized to use.


In [ ]:
# Dhaka Flex Plus — free Colab APK builder
# Runs on Google's temporary Linux VM, not on your phone.
from pathlib import Path
import os, sys, subprocess, shutil, urllib.request, time, zipfile, hashlib, traceback

ROOT=Path("/content")
PROJECT=ROOT/"dhaka-flex-plus"
SDK=ROOT/"android-sdk-dhaka-flex"
GRADLE_VERSION="8.11.1"
GRADLE=ROOT/("gradle-"+GRADLE_VERSION)
LOG=ROOT/"DhakaFlexPlus-Colab-Build.log"
APK=PROJECT/"app/build/outputs/apk/debug/app-debug.apk"
SDK_TOOLS_URL="https://dl.google.com/android/repository/commandlinetools-linux-15859902_latest.zip"
GRADLE_URL="https://services.gradle.org/distributions/gradle-8.11.1-bin.zip"
GRADLE_SHA256="f397b287023acdba1e9f6fc5ea72d22dd63669d59ed4a289a29b1a76eee151c6"

def run(cmd, cwd=None, env=None, input=None, live=False):
    label=" ".join(str(x) for x in cmd)
    print("\n▶", label, flush=True)
    if live:
        with open(LOG,"a",encoding="utf-8") as log:
            log.write("\n> "+label+"\n")
            p=subprocess.Popen(cmd,cwd=cwd,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
            for line in p.stdout:
                log.write(line)
                print(line,end="",flush=True)
            rc=p.wait()
            if rc:
                raise subprocess.CalledProcessError(rc,cmd)
            return
    subprocess.run(cmd,cwd=cwd,env=env,text=True,input=input,check=True)

def download(url,target):
    if target.exists() and zipfile.is_zipfile(target):
        print("Cached:",target.name)
        return
    target.parent.mkdir(parents=True,exist_ok=True)
    last=None
    for attempt in range(3):
        try:
            print("Downloading:",url,flush=True)
            with urllib.request.urlopen(urllib.request.Request(url,headers={"User-Agent":"DhakaFlexPlus-Colab-Builder/1.0"}),timeout=180) as src, open(target,"wb") as dst:
                shutil.copyfileobj(src,dst)
            if not zipfile.is_zipfile(target):
                raise ValueError("Downloaded file is not a ZIP archive")
            return
        except Exception as ex:
            last=ex
            target.unlink(missing_ok=True)
            print("Download attempt failed:",repr(ex))
            time.sleep(3*(attempt+1))
    raise RuntimeError("Could not download required tool: "+url) from last

def ensure_java_17():
    result=subprocess.run(["java","-version"],text=True,capture_output=True)
    if result.returncode!=0 or 'version "17.' not in result.stderr and 'openjdk 17.' not in result.stderr:
        run(["apt-get","update","-qq"])
        run(["apt-get","install","-y","-qq","openjdk-17-jdk-headless"])
    home=Path("/usr/lib/jvm/java-17-openjdk-amd64")
    if not home.exists():
        raise RuntimeError("Java 17 not found after installation")
    os.environ["JAVA_HOME"]=str(home)
    os.environ["PATH"]=str(home/"bin")+os.pathsep+os.environ["PATH"]
    run(["java","-version"])

def ensure_android_sdk():
    sdkmanager=SDK/"cmdline-tools/latest/bin/sdkmanager"
    if not sdkmanager.exists():
        archive=ROOT/"dhaka-android-commandlinetools.zip"
        download(SDK_TOOLS_URL,archive)
        tmp=ROOT/"dhaka-sdk-tool-unpack"
        if tmp.exists(): shutil.rmtree(tmp)
        tmp.mkdir(parents=True)
        with zipfile.ZipFile(archive) as z: z.extractall(tmp)
        original=tmp/"cmdline-tools"
        if not original.exists(): raise RuntimeError("Unexpected Android SDK tools archive layout")
        sdkmanager.parent.parent.parent.mkdir(parents=True,exist_ok=True)
        shutil.move(str(original),str(SDK/"cmdline-tools/latest"))
        shutil.rmtree(tmp)
    for binary in (SDK/"cmdline-tools/latest/bin").iterdir():
        if binary.is_file():
            binary.chmod(binary.stat().st_mode | 0o111)
    env=os.environ.copy()
    env["ANDROID_HOME"]=str(SDK)
    env["ANDROID_SDK_ROOT"]=str(SDK)
    env["PATH"]=str(sdkmanager.parent)+os.pathsep+env["PATH"]
    os.environ.update({k:env[k] for k in ("ANDROID_HOME","ANDROID_SDK_ROOT")})
    # Feed "y" explicitly to accept the Android SDK licenses.
    run([str(sdkmanager),"--sdk_root="+str(SDK),"--licenses"],env=env,input="y\n"*120)
    run([str(sdkmanager),"--sdk_root="+str(SDK),
         "platforms;android-36","build-tools;35.0.0","platform-tools"],env=env)
    return env

def ensure_gradle():
    executable=GRADLE/"bin/gradle"
    if executable.exists():
        executable.chmod(executable.stat().st_mode | 0o111)
        return executable
    archive=ROOT/"gradle-8.7-bin.zip"
    download(GRADLE_URL,archive)
    digest=hashlib.sha256(archive.read_bytes()).hexdigest()
    if digest != GRADLE_SHA256:
        archive.unlink(missing_ok=True)
        raise RuntimeError("Gradle archive checksum mismatch")
    with zipfile.ZipFile(archive) as z: z.extractall(ROOT)
    executable.chmod(executable.stat().st_mode | 0o111)
    return executable

try:
    print("=== Dhaka Flex Plus v0.6.1 — Cloud APK build ===")
    print("No GitHub Actions, payment card, or Google Drive required.")
    if PROJECT.exists(): shutil.rmtree(PROJECT)
    run(["git","clone","--depth","1","--branch","main",
         "https://github.com/fantominsider/dhaka-flex-plus.git",str(PROJECT)])
    ensure_java_17()
    env=ensure_android_sdk()
    gradle=ensure_gradle()
    env["JAVA_HOME"]=os.environ["JAVA_HOME"]
    env["ANDROID_HOME"]=str(SDK)
    env["ANDROID_SDK_ROOT"]=str(SDK)
    env["GRADLE_USER_HOME"]=str(ROOT/".gradle-dhaka-flex")
    run([str(gradle),"--no-daemon","--console=plain","--stacktrace",":app:assembleDebug"],
        cwd=PROJECT,env=env,live=True)
    if not APK.is_file(): raise FileNotFoundError("Gradle reported success but APK was not found")
    if not zipfile.is_zipfile(APK): raise ValueError("APK is not a valid ZIP container")
    digest=hashlib.sha256(APK.read_bytes()).hexdigest()
    print("\n✅ BUILD SUCCESS",flush=True)
    print("APK:",APK)
    print("Size: %.1f MiB"%(APK.stat().st_size/1024**2))
    print("SHA256:",digest)
    from google.colab import files
    files.download(str(APK))
except Exception as e:
    print("\n❌ BUILD DID NOT COMPLETE:",str(e))
    print("No usable APK was generated. Diagnostic log follows.")
    print(traceback.format_exc())
    if LOG.exists():
        print("\n--- Last 80 build-log lines ---")
        print("\n".join(LOG.read_text(errors="replace").splitlines()[-80:]))
        from google.colab import files
        files.download(str(LOG))
    raise
